In [ ]:
# Attention Mask : 실제 토큰 1 / 0 패딩
# token type ids (Segment ids) : 두개의 문장 (A/B) 구성될 때 각 토큰이 어느 문장에 속하는지 알려주는 임베딩
# CLS Token Pooling : [CLS] + token + [SEP]

In [3]:
#1. .BERT Tokenizer : 단어를 의미있는 조각 (subword)로 나눕니다. unbelivable -> un ##believ ##able
from transformers import BertTokenizer
# 토크나이져 로드
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
sentence = [
    "Hello, how are you?",
    "I am fine, thank you!"
]
for sentence in sentence:
    # 토큰화
    tokens = tokenizer.tokenize(sentence)
    print(f'원문 : {sentence}')
    print(f'토큰 : {tokens}')   

    # ID 변환
    ids = tokenizer.convert_tokens_to_ids(tokens)
    print(f'ID : {ids}')

    # 역변환
    decoded_string = tokenizer.decode(ids)
    print(f'역변환 : {decoded_string}\n')



원문 : Hello, how are you?
토큰 : ['hello', ',', 'how', 'are', 'you', '?']
ID : [7592, 1010, 2129, 2024, 2017, 1029]
역변환 : hello, how are you?

원문 : I am fine, thank you!
토큰 : ['i', 'am', 'fine', ',', 'thank', 'you', '!']
ID : [1045, 2572, 2986, 1010, 4067, 2017, 999]
역변환 : i am fine, thank you!



In [ ]:
# 2. Attention Mask : 실제단어 1  , 패딩은 0
from transformers import BertTokenizer
# 토크나이져 로드
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
sentences = [
    'short sentence',
    'Thiis is a much longer sentence with more words'
]
# 여러문장을 한꺼번에 토크나이징하고 가장 긴 문장길이에 맞춰 자동 패딩 수행
encoded = tokenizer(
    sentences,
    padding=True,
    return_tensors='pt'
)
encoded

{'input_ids': tensor([[  101,  2054,  2003,  2115,  2171,  1029,   102,     0,     0],
        [  101,  2026,  2171,  2003,  8836, 16480,  2063,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [11]:
#3. Attention Mask : 어떤 토큰에 집중해야하는지 모델에게 알려줍니다.
    # - 실제단어 1, 패딩은 0
from re import A
import token
import torch
from transformers import BertTokenizer
# 토크나이져 로드
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
sentences_A = "The weather is nice today."
sentences_B = [
    "Yes, it is a beautiful day.",
    "I love sunny days!"
]
# 두 문장을 하나의 입력으로 인코딩
encoded = tokenizer(
    sentences_A,
    sentences_B,
    padding=True,
    truncation=True,
    return_tensors="pt"
)
print(encoded)
tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
for token, token_id, type_id in zip(tokens, encoded['input_ids'][0], encoded['token_type_ids'][0]):
    segment = '문장 A' if type_id == 0 else '문장 B'
    if token == '[SEP]':
        segment = '구분자 [SEP]'
    elif token == '[CLS]':
        segment = '시작 토큰 [CLS]'
    print(f'Token: {token:20s} | Token ID: {token_id.item():6d} | Type ID: {type_id.item():6d} | Segment: {segment}')

{'input_ids': tensor([[ 101, 1996, 4633, 2003, 3835, 2651, 1012,  102,  100,  100,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Token: [CLS]                | Token ID:    101 | Type ID:      0 | Segment: 시작 토큰 [CLS]
Token: the                  | Token ID:   1996 | Type ID:      0 | Segment: 문장 A
Token: weather              | Token ID:   4633 | Type ID:      0 | Segment: 문장 A
Token: is                   | Token ID:   2003 | Type ID:      0 | Segment: 문장 A
Token: nice                 | Token ID:   3835 | Type ID:      0 | Segment: 문장 A
Token: today                | Token ID:   2651 | Type ID:      0 | Segment: 문장 A
Token: .                    | Token ID:   1012 | Type ID:      0 | Segment: 문장 A
Token: [SEP]                | Token ID:    102 | Type ID:      0 | Segment: 구분자 [SEP]
Token: [UNK]                | Token ID:    100 | Type ID:      1 | Segment: 문장 B
Token: [UNK]                | Token ID:  

In [ ]:
# token_type_ids : 두 문장을 입력할때 첫번째 ,두번째 구분
from transformers import BertTokenizer
# 토크나이져 로드
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
sentence_A = "The weather is nice"
sentence_B = "Let's go for a walk"
# 두 문장을 하나의 입력으로 인코딩
encoded = tokenizer(
    sentence_A,
    sentence_B,
    padding=True,
    return_tensors="pt"
)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
for token,token_id, type_id in zip(tokens, encoded["input_ids"][0],encoded["token_type_ids"][0]):
  segment = "문장 A" if type_id == 0 else "문장 B"
  if token == "[SEP]":
    segment = "구분자"
  elif token == "[CLS]":
    segment = "시작"
  print(f'{token:20s} {token_id.item():6d} {type_id.item():6d} ({segment})')

In [ ]:
# [CLS] Token Pooling : BERT 첫번째 토큰 [CLS] 문서 전체의 요약 => 분류 작업을 할때
# 이 토큰의 출력만 가져와서 분류기(classifier)에 연결
import torch
from transformers import BertTokenizer, BertModel
# 토크나이져 로드
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
sentence = "BERT is amazing for NLP tasks!"
# 인코딩
inputs =  tokenizer(sentence,return_tensors='pt')
# BERT 통과
with torch.no_grad():
  outputs = model(**inputs)
#   출력 형태 확인
last_hidden_state = outputs.last_hidden_state
print(f'입력문장 : {sentence}')
print(f'last_hidden_state 형태 : {last_hidden_state.shape}')
print(f'batch_size = 1 sequence_length = {last_hidden_state.shape[1]} \
  hidden_size = {last_hidden_state.shape[2]}')
# [CLS]토큰 추출
cls_embedding = last_hidden_state[:, 0, :]
print(f'cls_embedding 형태 : {cls_embedding.shape}')
# 분류기 (2-class)
classifier = torch.nn.Linear(768,2)
logits = classifier(cls_embedding)  # (1,2)  (batch, class개수) [[0.85,0.65]]
probs = torch.softmax(logits, dim=-1)
print(f'logits : {logits}')
print(f'probs : {probs}')
print(f'predicted class : {torch.argmax(probs).item()}')

입력 문장 : BERT is amazing. It is created by Google. 
last_hidden_state 크기 : torch.Size([1, 12, 768])
batch_size = 1 sequence_length = 12 hidden_size = 768
cls_embedding 형태 : torch.Size([1, 768])
logits : tensor([[-0.4774, -0.0875]], grad_fn=<AddmmBackward0>)
probs : tensor([[0.4037, 0.5963]], grad_fn=<SoftmaxBackward0>)
predicted class : 1


In [ ]:
# 미세 조정 학습 Fine-turning
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
texts = [
    "This movie is fantastic!",
    "Terrible film, waste of time.",
    "Amazing plot and great acting.",
    "Boring and predictable."
]
labels = [1, 0, 1, 0]  # 1=positive, 0=negative

# 토크나이져 모델
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# 모델
model = BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=2)
# 데이터셋
class SimpleDataset(Dataset):
  def __init__(self, texts, labels):
    self.encodings = tokenizer(texts, truncation=True, padding=True, return_tensors='pt')
    self.labels = labels
  def __getitem__(self, idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx])
    return item
  def __len__(self):
    return len(self.labels)
dataset = SimpleDataset(texts,labels)
loader = DataLoader(dataset, batch_size=2)
# 학습설정
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
optimizer = AdamW(model.parameters(), lr=1e-5)
# 미세조정
model.train()
for epoch in range(20):
  total_loss = 0
  for batch in loader:
    optimizer.zero_grad()
    inputs = { k:v.to(device) for k,v in batch.items() if k != 'labels'}
    labels = batch['labels'].to(device)
    outputs = model(**inputs, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  print(f'epoch : {epoch+1}, loss : {total_loss/len(loader)}')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
# 추론
model.eval() # 평가모드
sample_sentences = [
"I am really disappointed with the result.",
"The service was terrible and not worth the money.",
"I don't like this product at all."
]
# 토큰화
inputs =  tokenizer(
    sample_sentences,
    truncation=True,
    padding=True,
    return_tensors='pt'
)
# gpu/cpu 설정
inputs = { k: v.to(device) for k,v in inputs.items()}
 # 추론
with torch.no_grad():
   outputs = model(**inputs)
   logits = outputs.logits
   probs = torch.softmax(logits, dim=-1)
   pred = torch.argmax(probs, dim=-1).detach().numpy()
   print(pred,probs)
# probs, pred  # 1=positive, 0=negative

[0 0 0] tensor([[0.5637, 0.4363],
        [0.6189, 0.3811],
        [0.5258, 0.4742]])
